# SLM QLoRA Indonesian — Multi-Experiment Runner

Refactored for **reproducible multi-seed, multi-strategy** academic experiments on explicit linguistic supervision for Indonesian Small Language Models.

This notebook automatically runs **15 independent experiments** (5 strategies × 3 seeds). Set `DRY_RUN = False` in **Cell 3** then click **Run All**.

| Strategy | `linguistic_mode` |
|---|---|
| QLoRA | `NONE` |
| Normalization | `NORMALIZATION_ONLY` |
| Register | `REGISTER_ONLY` |
| Morphology | `MORPH_ONLY` |
| LiSA | `FULL_PIPELINE` |

Seeds: `[42, 123, 456]` → **15 total runs**

## Cell 1 — Install Dependencies

In [ ]:
!pip install torch torchvision torchaudio torchao --index-url https://download.pytorch.org/whl/cu121
!pip install -q transformers==5.6.2 accelerate==1.13.0
!pip install -q "peft>=0.19.1" "bitsandbytes>=0.49.2"
!pip install -q trl==0.19.0
!pip install -q "datasets<4.0.0" "evaluate>=0.4.6"
!pip install -q scikit-learn stanza

## Cell 2 — Imports & GPU Check

In [ ]:
import os, gc, json, random, time, collections, re, string, traceback
from datetime import datetime

import numpy as np
import pandas as pd
from tqdm import tqdm
import torch
from sklearn.metrics import accuracy_score, f1_score
from datasets import load_dataset, Dataset, concatenate_datasets
from transformers import (
    AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig,
    set_seed as hf_set_seed,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
from trl import SFTTrainer, SFTConfig

print("=" * 50)
print("GPU CHECK")
print("=" * 50)
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        vram_gb = props.total_memory / (1024 ** 3)
        print(f"GPU {i}: {props.name} \u2014 {vram_gb:.1f} GB VRAM")
    print(f"\nTotal GPU: {torch.cuda.device_count()}")
    print(f"CUDA version: {torch.version.cuda}")
else:
    print("\u274c No GPU found.")

## Cell 3 — Central Configuration

All experiment parameters are defined here. **Adjust `output_root` and `wiki_dataset_dir` before running.**

In [ ]:
# ============================================================
# CENTRAL CONFIGURATION
# ============================================================

CONFIG = {
    "model_name": "Qwen/Qwen2-1.5B",

    # \u2699\ufe0f Set output_root to your storage path
    # Kaggle:       "/kaggle/working/Riset_QLoRA/runs"
    # Google Colab: "/content/drive/MyDrive/Riset_QLoRA/runs"
    # Supercomputer: "/scratch/<username>/riset-brow/runs"
    "output_root": "Riset_QLoRA/runs",

    "wiki_dataset_dir": "datasets/dataset_tambahan.csv",

    # QLoRA parameters \u2014 DO NOT CHANGE without justification
    "lora_r": 32,
    "lora_alpha": 128,
    "lora_dropout": 0.05,
    "target_modules": "all-linear",

    # Training parameters \u2014 DO NOT CHANGE without justification
    "max_seq_length": 512,
    "batch_size": 2,
    "grad_acc_steps": 8,       # Effective batch size = 16
    "lr": 2e-5,
    "epochs": 3,

    "train_samples": 8000,
    "eval_limit": 150,

    # Research switches \u2014 DO NOT CHANGE
    "vocab_adaptation": False,
}

# Only these 5 strategies are run. NORM_MORPH is excluded.
EXPERIMENTS = {
    "QLoRA":         "NONE",
    "Normalization": "NORMALIZATION_ONLY",
    "Register":      "REGISTER_ONLY",
    "Morphology":    "MORPH_ONLY",
    "LiSA":          "FULL_PIPELINE",
}

# DATA_SEED: fixed for dataset sampling (same data across all runs)
# SEEDS: training seeds (varied per run)
DATA_SEED = 42
SEEDS = [42, 123, 456]

# DRY_RUN = True  -> preview plan, no training
# DRY_RUN = False -> run experiments
DRY_RUN = False

print("\u2705 Configuration loaded.")
print(f"   Model:       {CONFIG['model_name']}")
print(f"   Output root: {CONFIG['output_root']}")
print(f"   Experiments: {list(EXPERIMENTS.keys())}")
print(f"   Seeds:       {SEEDS}")
print(f"   Total runs:  {len(EXPERIMENTS) * len(SEEDS)}")
print(f"   DRY_RUN:     {DRY_RUN}")

## Cell 4 — Linguistic Preprocessing Utilities

Downloads Stanza and defines all linguistic annotation functions. Runs **once per session**.

In [ ]:
import stanza

print("\u23f3 Downloading and loading Stanza Indonesian model...")
stanza.download('id', processors='tokenize,pos,lemma')
nlp_id = stanza.Pipeline(
    lang='id', processors='tokenize,pos,lemma', use_gpu=True, verbose=False,
)
print("\u2705 Stanza ready.")

# Normalization dictionary (Kamus Alay) — Source: Salsabila et al. (IALP 2018)
KAMUS_ALAY = {
    "gw": "saya", "gue": "saya", "gua": "saya",
    "lo": "kamu", "lu": "kamu", "elo": "kamu",
    "dy": "dia", "dya": "dia",
    "yg": "yang", "yng": "yang",
    "dgn": "dengan", "dg": "dengan",
    "tdk": "tidak", "ga": "tidak", "gak": "tidak",
    "nggak": "tidak", "ngga": "tidak", "enggak": "tidak",
    "udah": "sudah", "udh": "sudah", "sdh": "sudah",
    "blm": "belum", "blum": "belum",
    "bgt": "banget", "bngt": "banget",
    "emg": "memang", "emang": "memang",
    "bkn": "bukan",
    "krn": "karena", "karna": "karena",
    "klo": "kalau", "klu": "kalau", "kl": "kalau",
    "tp": "tapi", "tpi": "tapi",
    "jd": "jadi", "jdi": "jadi",
    "sm": "sama", "brs": "bersama",
    "bs": "bisa", "bsa": "bisa",
    "sdg": "sedang", "lg": "lagi",
    "hrs": "harus",
    "jgn": "jangan", "jgnn": "jangan",
    "utk": "untuk", "tuk": "untuk",
    "dr": "dari", "dri": "dari", "pd": "pada",
    "ny": "nya", "na": "nya",
    "aja": "saja", "aj": "saja",
    "deh": "deh", "dong": "dong", "sih": "sih", "nih": "ini", "tuh": "itu",
    "tsb": "tersebut",
    "dll": "dan lain-lain", "dsb": "dan sebagainya", "dkk": "dan kawan-kawan",
    "stlh": "setelah", "sblm": "sebelum",
    "skrg": "sekarang", "skrang": "sekarang",
    "msh": "masih", "jg": "juga",
    "sy": "saya", "aku": "aku", "km": "kamu", "mrk": "mereka", "kt": "kita",
    "oke": "baik", "ok": "baik", "bgus": "bagus", "bgs": "bagus",
    "gimana": "bagaimana", "gmn": "bagaimana",
    "kenapa": "mengapa", "knp": "mengapa",
    "kapan": "kapan", "kpn": "kapan",
    "dimana": "di mana", "dmn": "di mana",
    "siapa": "siapa", "spa": "siapa",
}
SINGKATAN_FORMAL = {
    "yth": "yang terhormat", "ttd": "ditandatangani", "hlm": "halaman",
    "no": "nomor", "tgl": "tanggal", "bpk": "bapak", "ibu": "ibu",
    "sdr": "saudara", "sdri": "saudari", "prof": "profesor", "drs": "doktorandus",
}
INDIKATOR_INFORMAL = set(KAMUS_ALAY.keys()) | {
    "wkwk", "haha", "hihi", "hehe", "xixi", "anjir", "anjay",
    "mantap", "keren", "baper", "bucin", "gabut", "ntar", "bentar",
    "emang", "gimana", "kayak", "banget", "nih", "dong", "tuh",
    "deh", "sih", "woy", "bro", "sis", "gan",
}


def normalize_text(text: str) -> str:
    words = str(text).split()
    out = []
    for w in words:
        lw = w.lower()
        if lw in KAMUS_ALAY:
            out.append(KAMUS_ALAY[lw])
        elif lw in SINGKATAN_FORMAL:
            out.append(SINGKATAN_FORMAL[lw])
        else:
            out.append(re.sub(r'(.)\1{2,}', r'\1\1', w))
    return " ".join(out)


def detect_register(text: str) -> str:
    words = str(text).lower().split()
    ratio = sum(1 for w in words if w in INDIKATOR_INFORMAL) / max(len(words), 1)
    return "[INFORMAL]" if ratio > 0.05 else "[FORMAL]"


def extract_morph_tags(text: str, max_tokens: int = 30) -> str:
    # IMPORTANT: must ONLY receive instruction text — never target response.
    doc = nlp_id(str(text)[:500])
    tags = []
    for sent in doc.sentences:
        for token in sent.words[:max_tokens]:
            tags.append(f"{token.text}/{token.upos}/{token.lemma}")
        if len(tags) >= max_tokens:
            break
    return " ".join(tags[:max_tokens])


print("\u2705 Linguistic preprocessing utilities defined.")

## Cell 5 — Seed & Model Utility Functions

In [ ]:
def set_all_seeds(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    hf_set_seed(seed)
    print(f"  \U0001f3b2 Seeds set to {seed}")


def load_fresh_base_model(config: dict):
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,
    )
    tokenizer = AutoTokenizer.from_pretrained(config["model_name"], trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"
    model = AutoModelForCausalLM.from_pretrained(
        config["model_name"], quantization_config=bnb_config,
        device_map={"": "cuda:0"}, dtype=torch.float16, trust_remote_code=True,
    )
    print(f"  \u2705 Fresh {config['model_name']} loaded.")
    return model, tokenizer


def apply_fresh_lora(model, config: dict):
    model = prepare_model_for_kbit_training(model)
    model.config.torch_dtype = torch.float16
    peft_config = LoraConfig(
        r=config["lora_r"], lora_alpha=config["lora_alpha"],
        target_modules=config["target_modules"], lora_dropout=config["lora_dropout"],
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(model, peft_config)
    for _, param in model.named_parameters():
        if param.dtype == torch.bfloat16:
            param.data = param.data.to(torch.float16)
    for _, buf in model.named_buffers():
        if buf.dtype == torch.bfloat16:
            buf.data = buf.data.to(torch.float16)
    print(f"  \u2705 Fresh LoRA adapter applied (r={config['lora_r']}, alpha={config['lora_alpha']}).")
    return model


def cleanup_run(trainer=None, model=None, tokenizer=None) -> None:
    if trainer is not None: del trainer
    if model is not None: del model
    if tokenizer is not None: del tokenizer
    gc.collect()
    torch.cuda.empty_cache()
    print("  \U0001f9f9 GPU memory released.")


def result_is_complete(run_output_dir: str) -> bool:
    return os.path.isfile(os.path.join(run_output_dir, "COMPLETED"))


def check_output_dir_safety(run_output_dir: str) -> None:
    if os.path.exists(run_output_dir) and os.listdir(run_output_dir):
        raise FileExistsError(
            f"Output directory already exists and is not empty:\n  {run_output_dir}\n"
            "Delete it manually or choose a different output_root to re-run."
        )


print("\u2705 Utility functions defined.")

## Cell 6 — Dataset Builder

Dataset composition is **fixed across all seeds** (controlled by `DATA_SEED`). Only training randomness differs between seeds.

In [ ]:
LABEL_MAP_NLI = {0: "ENTAILMENT", 1: "NEUTRAL", 2: "CONTRADICTION"}


def build_linguistic_prompt(instruction: str, output: str, mode: str, tokenizer) -> dict:
    # MORPHOLOGY FIX: extract_morph_tags is called on instruction ONLY.
    # Target-response text is NEVER passed to extract_morph_tags().
    # Research design: Instruction -> Linguistic analysis -> Enriched instruction -> Model -> Response
    inst_processed = instruction
    out_processed = output

    if mode in ["NORMALIZATION_ONLY", "FULL_PIPELINE", "NORM_MORPH"]:
        inst_processed = normalize_text(instruction)
        out_processed = normalize_text(output)

    register_tag = ""
    if mode in ["REGISTER_ONLY", "FULL_PIPELINE"]:
        register_tag = detect_register(instruction) + "\n"

    morph_tag = ""
    if mode in ["MORPH_ONLY", "FULL_PIPELINE", "NORM_MORPH"]:
        morph_line = extract_morph_tags(inst_processed)  # <- instruction only
        if morph_line:
            morph_tag = f"[MORPH: {morph_line}]\n"

    user_msg = f"{register_tag}{morph_tag}Instruksi:\n{inst_processed}"
    messages = [
        {"role": "user", "content": user_msg},
        {"role": "assistant", "content": out_processed},
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False)}


def build_training_dataset(linguistic_mode: str, config: dict, tokenizer,
                           data_seed: int = DATA_SEED):
    # Dataset composition is fixed across all seeds via DATA_SEED.
    # IndoNLI is kept plain (no linguistic intervention) by design.
    print(f"  \u23f3 Building training dataset for mode: {linguistic_mode}...")

    raw_bactrian = (
        load_dataset("MBZUAI/Bactrian-X", "id", split="train")
        .shuffle(seed=data_seed).select(range(config["train_samples"]))
    )
    raw_wiki_df = pd.read_csv(config["wiki_dataset_dir"]).sample(n=3500, random_state=data_seed)
    raw_nli = (
        load_dataset("afaji/indonli", split="train", trust_remote_code=True)
        .shuffle(seed=data_seed).select(range(7000))
    )

    b_processed = []
    for row in tqdm(raw_bactrian, desc="  Bactrian NLP"):
        inst = (row.get("instruction") or "").strip()
        inp  = (row.get("input") or "").strip()
        full_inst = f"{inst}\n{inp}" if inp else inst
        b_processed.append(build_linguistic_prompt(
            full_inst, (row.get("output") or "").strip(), linguistic_mode, tokenizer))
    b_ds = Dataset.from_list(b_processed)

    w_processed = []
    for _, row in tqdm(raw_wiki_df.iterrows(), total=len(raw_wiki_df), desc="  Wiki NLP"):
        w_processed.append(build_linguistic_prompt(
            str(row.get("instruction", "")), str(row.get("output", "")), linguistic_mode, tokenizer))
    w_ds = Dataset.from_list(w_processed)

    # IndoNLI: ALWAYS kept plain — no linguistic intervention (preserves logic-training integrity)
    n_processed = []
    for row in tqdm(raw_nli, desc="  IndoNLI Formatting"):
        p = row.get("premise", "")
        h = row.get("hypothesis", "")
        label = LABEL_MAP_NLI.get(row.get("label", 1), "NEUTRAL")
        user_msg = (
            "Diberikan sebuah Premis dan Hipotesis. Tentukan hubungan logis di antara keduanya.\n"
            "Jawab HANYA dengan satu kata: ENTAILMENT, NEUTRAL, atau CONTRADICTION.\n\n"
            f"Premis: {p}\nHipotesis: {h}"
        )
        msgs = [{"role": "user", "content": user_msg}, {"role": "assistant", "content": label}]
        n_processed.append({"text": tokenizer.apply_chat_template(msgs, tokenize=False)})
    n_ds = Dataset.from_list(n_processed)

    combined = concatenate_datasets([b_ds, n_ds, w_ds]).shuffle(seed=data_seed)
    split = combined.train_test_split(test_size=500, seed=data_seed)
    print(f"  \u2705 Dataset ready: {len(split['train'])} train / {len(split['test'])} eval")
    return split["train"], split["test"]


print("\u2705 Dataset builder defined.")

## Cell 7 — Evaluation Utilities

In [ ]:
def normalize_qa_answer(s):
    exclude = set(string.punctuation)
    return " ".join("".join(c for c in str(s).lower() if c not in exclude).split())

def qa_exact_match(p, g): return 1.0 if normalize_qa_answer(p) == normalize_qa_answer(g) else 0.0

def qa_f1_score(p, g):
    pt = normalize_qa_answer(p).split()
    gt = normalize_qa_answer(g).split()
    if not pt and not gt: return 1.0
    if not pt or not gt: return 0.0
    common = collections.Counter(pt) & collections.Counter(gt)
    ns = sum(common.values())
    if ns == 0: return 0.0
    pr = ns / len(pt); rc = ns / len(gt)
    return (2 * pr * rc) / (pr + rc)


def evaluate_batched_final(model, tokenizer, dataset, task_name, linguistic_mode, batch_size=8):
    # NOTE: linguistic_mode must be the mode for THIS specific run.
    model.eval()
    results = []
    tag_prefix = ""
    if linguistic_mode in ["REGISTER_ONLY", "FULL_PIPELINE"]:
        tag_prefix = "[INFORMAL]\n" if task_name == "smsa" else "[FORMAL]\n"

    for i in tqdm(range(0, len(dataset), batch_size), desc=f"  Eval {task_name}"):
        batch = dataset.select(range(i, min(i + batch_size, len(dataset))))
        prompts = []
        for s in batch:
            if task_name == "indonli":
                p = (
                    "Diberikan sebuah Premis dan Hipotesis. Tentukan hubungan logis di antara keduanya.\n"
                    "Jawab HANYA dengan satu kata: ENTAILMENT, NEUTRAL, atau CONTRADICTION.\n\n"
                    f"Premis: {s['premise']}\nHipotesis: {s['hypothesis']}"
                )
            elif task_name == "copal":
                qt = str(s.get("question", "")).lower()
                pertanyaan = ("Apa penyebab dari situasi tersebut?" if qt == "cause" else
                              "Apa akibat dari situasi tersebut?" if qt == "effect" else
                              "Manakah pilihan yang paling masuk akal?")
                p = (f"{tag_prefix}Instruksi:\nSituasi: {s['premise']}\nPertanyaan: {pertanyaan}\n"
                     f"A. {s['choice1']}\nB. {s['choice2']}\nJawaban (A/B):")
            elif task_name == "qa":
                pt = " ".join(s.get("passage", [])).replace(" ,", ",").replace(" .", ".")
                qt = " ".join(s.get("question", [])).replace(" ,", ",").replace(" ?", "?")
                p = (f"{tag_prefix}Instruksi:\nBerdasarkan teks berikut, jawablah pertanyaan dengan singkat.\n\n"
                     f"Teks: {pt}\nPertanyaan: {qt}\n\nJawaban:")
            elif task_name == "indoculture":
                opts = s.get("options", [])
                opts_text = "\n".join(opts) if isinstance(opts, list) else str(opts)
                p = (f"{tag_prefix}Instruksi:\nPilih jawaban paling tepat berdasarkan konteks budaya Indonesia.\n\n"
                     f"Konteks: {s['context']}\nPilihan:\n{opts_text}\n\nJawaban (A, B, atau C):")
            else:  # smsa
                p = (f"{tag_prefix}Instruksi:\nAnalisis sentimen. Jawab HANYA: POSITIF, NETRAL, atau NEGATIF.\n\n"
                     f"Teks: {s['text']}\nJawaban:")
            prompts.append(tokenizer.apply_chat_template(
                [{"role": "user", "content": p}], tokenize=False, add_generation_prompt=True))

        inputs = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True).to(model.device)
        with torch.no_grad():
            t0 = time.perf_counter()
            outs = model.generate(**inputs, max_new_tokens=10, do_sample=False,
                                  pad_token_id=tokenizer.eos_token_id)
            t1 = time.perf_counter()
        for out in outs:
            gen = tokenizer.decode(out[inputs["input_ids"].shape[1]:], skip_special_tokens=True).upper()
            results.append({"pred": gen, "latency": (t1 - t0) / len(batch)})
    return results


def extract_hybrid_prediction(task_name, gen_text):
    g = str(gen_text).strip().upper()
    if task_name in ["copal", "indoculture"]:
        m = re.search(r'^([A-C])[\.\s\(\:]', g)
        if m: return m.group(1)
        if len(g) == 1 and g in "ABC": return g
        for lbl in "ABC":
            if f"JAWABAN: {lbl}" in g or f"ADALAH {lbl}" in g or f"PILIHAN {lbl}" in g:
                return lbl
    elif task_name == "indonli":
        if "SESUAI" in g or "ENTAILMENT" in g: return "SESUAI"
        if "BERTENTANGAN" in g or "CONTRADICTION" in g: return "BERTENTANGAN"
        if "NETRAL" in g or "NEUTRAL" in g: return "NETRAL"
    elif task_name == "smsa":
        if "POSITIF" in g or "POSITIVE" in g: return "POSITIF"
        if "NEGATIF" in g or "NEGATIVE" in g: return "NEGATIF"
        if "NETRAL" in g or "NEUTRAL" in g: return "NETRAL"
    elif task_name == "qa":
        ct = g.split("\n")[0].strip()
        m = re.search(r'^[A-C]\s*[\(\.\:]\s*(.*)', ct)
        if m: return m.group(1).replace(")", "").strip()
        return ct
    return "UNKNOWN"


def map_hybrid_label(task_name, sample):
    if task_name == "qa":
        pt = sample.get("passage", [])
        ls = sample.get("seq_label", [])
        ans = " ".join(t for t, l in zip(pt, ls) if l in [1, 2]).strip().upper()
        return ans.replace(" ,", ",").replace(" .", ".") if ans else "UNKNOWN"
    raw = str(sample.get("label", sample.get("answer", sample.get("target", "-1")))).strip().upper()
    if task_name == "copal":   return "A" if raw == "0" else "B"
    if task_name == "indoculture":
        m = {"0": "A", "1": "B", "2": "C"}
        return raw if raw in "ABC" else m.get(raw, "UNKNOWN")
    if task_name == "indonli":
        m = {"0": "SESUAI", "1": "NETRAL", "2": "BERTENTANGAN"}
        return m.get(raw, "UNKNOWN")
    if task_name == "smsa":
        m = {"0": "POSITIF", "1": "NETRAL", "2": "NEGATIF"}
        return raw if raw in ["POSITIF", "NETRAL", "NEGATIF"] else m.get(raw, "UNKNOWN")
    return raw


print("\u2705 Evaluation utilities defined.")

## Cell 8 — Load Evaluation Datasets

Evaluation datasets are loaded **once** and shared across all 15 runs.

In [ ]:
print("Loading evaluation datasets...")
eval_tasks = {}

try:
    eval_tasks["copal"] = load_dataset("haryoaw/COPAL", split="test", trust_remote_code=True)
    print(f"  \u2705 COPAL: {len(eval_tasks['copal'])} samples")
except Exception as e:
    print(f"  \u26a0\ufe0f COPAL load failed: {e}")
    eval_tasks["copal"] = None

try:
    eval_tasks["indonli"] = load_dataset("afaji/indonli", split="test", trust_remote_code=True)
    print(f"  \u2705 IndoNLI: {len(eval_tasks['indonli'])} samples")
except Exception as e:
    print(f"  \u26a0\ufe0f IndoNLI load failed: {e}")
    eval_tasks["indonli"] = None

try:
    eval_tasks["indoculture"] = load_dataset("sabilmakbar/indo_culture", split="test", trust_remote_code=True)
    print(f"  \u2705 IndoCulture: {len(eval_tasks['indoculture'])} samples")
except Exception as e:
    print(f"  \u26a0\ufe0f IndoCulture load failed: {e}")
    eval_tasks["indoculture"] = None

print("\n\u2705 Evaluation datasets loaded.")

## Cell 9 — Single Experiment Run Function

`run_experiment()` encapsulates one complete training + evaluation cycle.

**Output structure:**
```
output_root/{strategy}_seed{seed}/
├── adapter/       <- saved LoRA adapter
├── results.json   <- evaluation results
├── ERROR.log      <- written only on failure
└── COMPLETED      <- written only after full success
```

In [ ]:
def run_experiment(strategy_name, linguistic_mode, seed, config, eval_tasks):
    run_name = f"{strategy_name}_seed{seed}"
    run_output_dir  = os.path.join(config["output_root"], run_name)
    results_dir     = os.path.join(config["output_root"], "results")
    result_file     = os.path.join(results_dir, f"{run_name}.json")
    adapter_dir     = os.path.join(run_output_dir, "adapter")
    completed_marker = os.path.join(run_output_dir, "COMPLETED")

    print("\n" + "=" * 60)
    print("STARTING RUN")
    print("=" * 60)
    print(f"  Strategy : {strategy_name}")
    print(f"  Mode     : {linguistic_mode}")
    print(f"  Seed     : {seed}")
    print(f"  Run name : {run_name}")
    print(f"  Output   : {run_output_dir}")
    print("=" * 60)

    # Resume: skip if already completed
    if result_is_complete(run_output_dir):
        print(f"  \u23ed\ufe0f  Skipping already-completed run: {run_name}")
        if os.path.isfile(result_file):
            with open(result_file) as f: return json.load(f)
        return {"run_name": run_name, "status": "skipped"}

    trainer = None; eval_model = None; tokenizer = None

    try:
        check_output_dir_safety(run_output_dir)
        os.makedirs(run_output_dir, exist_ok=True)
        os.makedirs(results_dir,    exist_ok=True)
        os.makedirs(adapter_dir,    exist_ok=True)

        # [1/6] Set seeds
        print("\n[1/6] Setting random seeds...")
        set_all_seeds(seed)

        # [2/6] Load fresh model
        print("\n[2/6] Loading fresh base model...")
        model, tokenizer = load_fresh_base_model(config)

        # [3/6] Build dataset
        print("\n[3/6] Building training dataset...")
        train_dataset, eval_dataset = build_training_dataset(
            linguistic_mode, config, tokenizer, data_seed=DATA_SEED
        )

        # [4/6] Apply fresh LoRA
        print("\n[4/6] Applying fresh LoRA adapter...")
        model = apply_fresh_lora(model, config)

        # [5/6] Train
        print("\n[5/6] Training...")
        training_args = SFTConfig(
            output_dir=run_output_dir,
            num_train_epochs=config["epochs"],
            per_device_train_batch_size=config["batch_size"],
            gradient_accumulation_steps=config["grad_acc_steps"],
            learning_rate=config["lr"],
            seed=seed,         # propagate training seed
            data_seed=seed,    # propagate data shuffle seed
            logging_steps=20, fp16=False, bf16=False,
            optim="paged_adamw_8bit", max_length=config["max_seq_length"],
            eval_strategy="steps", 
            # save_total_limit=2,
            eval_steps=100, 
            # save_strategy="steps", 
            # save_steps=100,
            report_to="none",
        )
        tokenizer.padding_side = "right"
        trainer = SFTTrainer(
            model=model, train_dataset=train_dataset, eval_dataset=eval_dataset,
            args=training_args, processing_class=tokenizer,
        )
        model.get_input_embeddings().weight.requires_grad = True
        trainer.train()

        trainer.save_model(adapter_dir)
        tokenizer.save_pretrained(adapter_dir)
        print(f"  \u2705 Adapter saved to: {adapter_dir}")

        # [6/6] Evaluate
        print("\n[6/6] Evaluating...")
        tokenizer.padding_side = "left"
        del model; del trainer; trainer = None
        gc.collect(); torch.cuda.empty_cache()

        bnb_eval = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,
        )
        base_model = AutoModelForCausalLM.from_pretrained(
            config["model_name"], quantization_config=bnb_eval,
            device_map="auto", trust_remote_code=True,
        )
        eval_model = PeftModel.from_pretrained(base_model, adapter_dir)
        eval_model.eval()

        task_results = {}
        copal_meta = {
            "culture":     {"1": {"correct": 0, "total": 0}, "0": {"correct": 0, "total": 0}},
            "terminology": {"1": {"correct": 0, "total": 0}, "0": {"correct": 0, "total": 0}},
        }

        for task_name, dataset in eval_tasks.items():
            if dataset is None:
                print(f"  \u26a0\ufe0f Skipping {task_name}: not available"); continue
            print(f"\n  \U0001f680 Evaluating {task_name.upper()}...")
            eval_subset = dataset.select(range(min(config["eval_limit"], len(dataset))))

            raw_results = evaluate_batched_final(
                model=eval_model, tokenizer=tokenizer, dataset=eval_subset,
                task_name=task_name, linguistic_mode=linguistic_mode,  # correct mode
                batch_size=config["batch_size"],
            )

            preds, labels, latencies = [], [], []
            per_sample = []; unknown_count = 0

            for i, res in enumerate(raw_results):
                pf = extract_hybrid_prediction(task_name, res["pred"])
                tl = map_hybrid_label(task_name, eval_subset[i])
                preds.append(pf); labels.append(tl); latencies.append(res["latency"])
                if pf == "UNKNOWN": unknown_count += 1
                per_sample.append({"id": i, "ground_truth": tl,
                                   "raw_model_output": res["pred"], "parsed_prediction": pf,
                                   "valid": pf != "UNKNOWN", "correct": pf == tl})
                if task_name == "copal":
                    ic = (pf == tl)
                    cv = str(eval_subset[i].get("Culture", "0"))
                    tv = str(eval_subset[i].get("Terminology", "0"))
                    if cv in copal_meta["culture"]:
                        copal_meta["culture"][cv]["total"] += 1
                        if ic: copal_meta["culture"][cv]["correct"] += 1
                    if tv in copal_meta["terminology"]:
                        copal_meta["terminology"][tv]["total"] += 1
                        if ic: copal_meta["terminology"][tv]["correct"] += 1

            if task_name == "qa":
                acc    = float(np.mean([qa_exact_match(p, l) for p, l in zip(preds, labels)]) * 100)
                mac_f1 = float(np.mean([qa_f1_score(p, l)    for p, l in zip(preds, labels)]) * 100)
            else:
                acc    = float(accuracy_score(labels, preds) * 100)
                mac_f1 = float(f1_score(labels, preds, average="macro", zero_division=0) * 100)

            avg_lat = float(np.mean(latencies))
            print(f"    \U0001f4ca Accuracy: {acc:.2f}%  |  Macro F1: {mac_f1:.2f}%  |  Unknown: {unknown_count}")
            task_results[task_name] = {"accuracy": acc, "macro_f1": mac_f1, "latency": avg_lat,
                                       "unknown_count": unknown_count, "per_sample": per_sample}

        run_result = {
            "run_name": run_name, "strategy": strategy_name,
            "linguistic_mode": linguistic_mode, "seed": seed,
            "model": config["model_name"], "epochs": config["epochs"],
            "learning_rate": config["lr"], "batch_size": config["batch_size"],
            "gradient_accumulation_steps": config["grad_acc_steps"],
            "max_seq_length": config["max_seq_length"],
            "lora_r": config["lora_r"], "lora_alpha": config["lora_alpha"],
            "vocab_adaptation": config["vocab_adaptation"],
            "task_results": task_results, "copal_metadata": copal_meta,
            "completed_at": datetime.now().isoformat(), "status": "success",
        }
        with open(result_file, "w") as f: json.dump(run_result, f, indent=2)
        with open(os.path.join(run_output_dir, "results.json"), "w") as f: json.dump(run_result, f, indent=2)
        with open(completed_marker, "w") as f: f.write("success")

        print("\n" + "=" * 60 + "\nCOMPLETED RUN\n" + "=" * 60)
        print(f"  Strategy : {strategy_name}\n  Seed     : {seed}\n  Status   : SUCCESS")
        print("=" * 60)
        return run_result

    except Exception as e:
        err = traceback.format_exc()
        print(f"\n\u274c RUN FAILED: {run_name}\n{err}")
        try:
            os.makedirs(run_output_dir, exist_ok=True)
            with open(os.path.join(run_output_dir, "ERROR.log"), "w") as f:
                f.write(f"Run: {run_name}\nTimestamp: {datetime.now().isoformat()}\n\n{err}")
        except Exception: pass
        return {"run_name": run_name, "status": "failed", "error": str(e)}

    finally:
        cleanup_run(trainer=trainer, model=eval_model, tokenizer=tokenizer)


print("\u2705 run_experiment() function defined.")

## Cell 10 — Result Aggregation Function

In [ ]:
def aggregate_all_results(output_root, experiments, seeds):
    results_dir = os.path.join(output_root, "results")
    all_rows = []
    print("\n" + "=" * 60 + "\nAGGREGATING RESULTS\n" + "=" * 60)

    for strategy_name in experiments:
        for seed in seeds:
            run_name = f"{strategy_name}_seed{seed}"
            rf = os.path.join(results_dir, f"{run_name}.json")
            if not os.path.isfile(rf):
                print(f"  \u26a0\ufe0f  Missing: {rf}"); continue
            with open(rf) as f: result = json.load(f)
            row = {"run_name": run_name, "strategy": strategy_name, "seed": seed,
                   "status": result.get("status", "unknown")}
            for tn, td in result.get("task_results", {}).items():
                row[f"{tn}_accuracy"] = td.get("accuracy")
                row[f"{tn}_macro_f1"] = td.get("macro_f1")
                row[f"{tn}_unknown"]  = td.get("unknown_count")
            all_rows.append(row)

    if not all_rows:
        print("  \u26a0\ufe0f  No results found."); return

    df = pd.DataFrame(all_rows)
    csv_path = os.path.join(output_root, "all_results.csv")
    df.to_csv(csv_path, index=False)
    print(f"  \U0001f4be all_results.csv: {csv_path}")
    print(f"  \U0001f4ca Rows: {len(df)} / expected: {len(experiments) * len(seeds)}")
    acc_cols = [c for c in df.columns if c.endswith("_accuracy")]
    print(); print(df[["run_name", "status"] + acc_cols].to_string(index=False))

    with open(os.path.join(output_root, "all_results.json"), "w") as f:
        json.dump(all_rows, f, indent=2)

    # Aggregate stats (mean +/- std)
    metric_cols = [c for c in df.columns if c.endswith("_accuracy") or c.endswith("_macro_f1")]
    agg_rows = []
    for sn in experiments:
        sdf = df[df["strategy"] == sn]
        for mc in metric_cols:
            vals = sdf[mc].dropna().tolist()
            if not vals: continue
            parts = mc.rsplit("_", 1)
            agg_rows.append({"strategy": sn,
                             "task":   parts[0] if len(parts) == 2 else mc,
                             "metric": parts[1] if len(parts) == 2 else "value",
                             "mean": float(np.mean(vals)),
                             "std":  float(np.std(vals)),
                             "n":    len(vals)})
    agg_df = pd.DataFrame(agg_rows)
    agg_path = os.path.join(output_root, "aggregate_results.csv")
    agg_df.to_csv(agg_path, index=False)
    print(f"\n  \U0001f4c8 aggregate_results.csv: {agg_path}")
    if not agg_df.empty: print(); print(agg_df.to_string(index=False))


print("\u2705 aggregate_all_results() function defined.")

## Cell 11 — Dry Run Preview

When `DRY_RUN = True`, prints the planned experiment matrix without training.

In [ ]:
if DRY_RUN:
    print("=" * 60)
    print("DRY RUN MODE \u2014 No training will occur")
    print("=" * 60)
    print("\nConfiguration:")
    for k, v in [("Model", CONFIG["model_name"]), ("Output root", CONFIG["output_root"]),
                 ("LoRA rank", CONFIG["lora_r"]), ("LoRA alpha", CONFIG["lora_alpha"]),
                 ("LR", CONFIG["lr"]), ("Epochs", CONFIG["epochs"]),
                 ("Effective batch", f"{CONFIG['batch_size']} x {CONFIG['grad_acc_steps']} = {CONFIG['batch_size']*CONFIG['grad_acc_steps']}"),
                 ("Max seq len", CONFIG["max_seq_length"]),
                 ("Vocab adapt", CONFIG["vocab_adaptation"]),
                 ("DATA_SEED", DATA_SEED), ("SEEDS", SEEDS)]:
        print(f"  {k:<16}: {v}")
    print()
    print("Planned experiments:")
    total = 0
    for strategy_name, linguistic_mode in EXPERIMENTS.items():
        for seed in SEEDS:
            total += 1
            rn = f"{strategy_name}_seed{seed}"
            rod = os.path.join(CONFIG["output_root"], rn)
            status = "[COMPLETED \u2014 will skip]" if result_is_complete(rod) else "[PENDING]"
            print(f"  {total:2d}. {rn:<30s}  {status}")
    print(f"\nTotal: {total} runs")
    print("\n\u2705 Set DRY_RUN = False in Cell 3 to execute.")
    print("=" * 60)
else:
    print("DRY_RUN = False \u2014 proceeding to master experiment loop.")

## Cell 12 — Master Experiment Loop

> **⚠️ This cell runs all 15 experiments automatically.**
>
> Only execute after confirming the dry run and setting `DRY_RUN = False` in Cell 3.

In [ ]:
if DRY_RUN:
    print("DRY_RUN = True \u2014 skipping master experiment loop.")
    print("Set DRY_RUN = False in Cell 3 to run experiments.")
else:
    successful_runs, failed_runs, skipped_runs = [], [], []
    total_runs = len(EXPERIMENTS) * len(SEEDS)
    current_run = 0

    for strategy_name, linguistic_mode in EXPERIMENTS.items():
        for seed in SEEDS:
            current_run += 1
            run_name = f"{strategy_name}_seed{seed}"
            print(f"\n{'#'*60}\n# Progress: {current_run}/{total_runs} \u2014 {run_name}\n{'#'*60}")

            result = run_experiment(
                strategy_name=strategy_name,
                linguistic_mode=linguistic_mode,
                seed=seed,
                config=CONFIG,
                eval_tasks=eval_tasks,
            )

            status = result.get("status", "unknown")
            if status == "success":
                successful_runs.append(run_name)
            elif status == "skipped":
                skipped_runs.append(run_name)
                successful_runs.append(run_name)
            else:
                failed_runs.append(run_name)

    print("\n" + "=" * 60 + "\nEXPERIMENT LOOP COMPLETE\n" + "=" * 60)
    print(f"  Successful runs : {len(successful_runs)} / {total_runs}")
    if skipped_runs:
        print(f"  Skipped (resume): {len(skipped_runs)}")
        for r in skipped_runs: print(f"    \u23ed\ufe0f  {r}")
    if failed_runs:
        print(f"  Failed runs     : {len(failed_runs)}")
        for r in failed_runs: print(f"    \u274c {r}")
    else:
        print("  No failed runs. \u2705")
    print("=" * 60)

## Cell 13 — Aggregate All Results

Run after the experiment loop completes to generate `all_results.csv` and `aggregate_results.csv`.

In [ ]:
if not DRY_RUN:
    aggregate_all_results(
        output_root=CONFIG["output_root"],
        experiments=EXPERIMENTS,
        seeds=SEEDS,
    )
else:
    print("DRY_RUN = True \u2014 skipping aggregation.")